ESERCIZIO

Esercitazione NER con BERT
1. Carica il modello 'Babelscape/wikineural-multilingual-ner' tramite Hugging Face
2. Implementa una funzione che estragga le entità di una stringa complessa contenente nomi di leggi e date storiche
3. Identiica quanti token WordPiece compontono l'entità 'Costituzione della Repubblica Italiana' e varifica se il modello assegna correttamente i tag Beginning e Inside a ogni frammento

Suggerimento: osserva bene come il tokenizer gestisce gli spazi e la maiuscole

In [3]:
#import os
#import numpy as np

# 1. CONFIGURAZIONE AMBIENTE
# Forza Keras 3 a usare PyTorch e comunica a Transformers di fare lo stesso
#os.environ["KERAS_BACKEND"] = "torch"

#import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# 2. SELEZIONE MODELLO REALISTICO
# Usiamo WikiNeural: 
# è un modello mBERT fine-tuned per il NER multilingua
# l'italiano è una delle lingue supportate
model_name = "Babelscape/wikineural-multilingual-ner"

print(f"--- Caricamento modello NER professionale: {model_name} ---")

# Caricamento Tokenizer e Modello con pesi NER reali
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Creazione Pipeline specifica per PyTorch (pt)
# 'aggregation_strategy=None' ci permette di vedere i singoli tag B- e I- per ogni token
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="none" , device="cpu")


# ==========================================
# 3. FUNZIONE DI ESTRAZIONE ENTITÀ (Punto 2 della traccia)
# ==========================================
def estrai_entita_complesse(testo: str):
    print(f"\n>>> ANALISI TESTO: {testo}")
    print("-" * 60)
    
    # Eseguiamo l'inferenza
    results = ner_pipeline(testo)
    
    print(f"{'Token (WordPiece)':<20} | {'Tag BIO':<8} | {'Confidenza':<10}")
    print("-" * 60)
    
    for res in results:
        # Pulizia estetica per i token WordPiece (rimuove il carattere di spazio Ġ o _)
        clean_word = res['word'].replace(" ", "")
        print(f"{clean_word:<20} | {res['entity']:<8} | {res['score']:.4f}")

# ==========================================
# 4. ANALISI SPECIFICA WORDPIECE (Punto 3 della traccia)
# ==========================================
def verifica_tag_bi(entita_target: str):
    print(f"\n--- VERIFICA DETTAGLIATA TAG B/I: '{entita_target}' ---")
    
    # Vediamo come il tokenizer spezza la stringa
    tokens_wp = tokenizer.tokenize(entita_target)
    print(f"Scomposizione WordPiece: {tokens_wp}")
    
    # Vediamo i tag assegnati
    risultati = ner_pipeline(entita_target)
    
    for r in risultati:
        tag = r['entity']
        word = r['word']
        desc = "Inizio (Beginning)" if tag.startswith("B-") else "Interno (Inside)"
        print(f"Token: {word:<12} | Tag: {tag:<6} | Significato: {desc}")

# ==========================================
# ESECUZIONE TEST REALE
# ==========================================

# Stringa complessa con leggi e date
stringa_test = (
    "Il 1° gennaio 1948 entrò in vigore la Costituzione della Repubblica Italiana, "
    "sostituendo lo Statuto Albertino firmato nel 1848."
)

# Eseguiamo l'estrazione
estrai_entita_complesse(stringa_test)

# Focus richiesto sulla Costituzione
verifica_tag_bi("Costituzione della Repubblica Italiana")

--- Caricamento modello NER professionale: Babelscape/wikineural-multilingual-ner ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3424.36it/s]



>>> ANALISI TESTO: Il 1° gennaio 1948 entrò in vigore la Costituzione della Repubblica Italiana, sostituendo lo Statuto Albertino firmato nel 1848.
------------------------------------------------------------
Token (WordPiece)    | Tag BIO  | Confidenza
------------------------------------------------------------
Cost                 | B-MISC   | 0.8966
##itu                | I-MISC   | 0.9699
##zione              | I-MISC   | 0.9733
della                | I-MISC   | 0.9703
Repubblica           | I-MISC   | 0.9401
Italiana             | I-MISC   | 0.9438
Stat                 | B-MISC   | 0.9244
##uto                | I-MISC   | 0.9804
Alberti              | I-MISC   | 0.9693
##no                 | I-MISC   | 0.9791

--- VERIFICA DETTAGLIATA TAG B/I: 'Costituzione della Repubblica Italiana' ---
Scomposizione WordPiece: ['Cost', '##itu', '##zione', 'della', 'Repubblica', 'Italiana']
Token: Cost         | Tag: B-MISC | Significato: Inizio (Beginning)
Token: ##itu        | Tag: I-MISC | S